# Notebook 3 — Supervised layer detectors (template: one detector × one attack)

**Pipeline position:** detector fitting, sharded for parallelism. Each run of this notebook fits
**one** supervised detector for **one** attack on **one** dataset, and saves its `.pkl` + `.npy`
for the ensemble. Copy & Edit on Kaggle, change the three lines in the **Run** cell, launch.

This is a **generic supervised-detector template** — it fits any of:
**KNN, Random Forest, AdaBoost, XGBoost, LightGBM** (set `DETECTOR` in the Run cell).
OCSVM and the LID/Maha selection have their own notebooks.

### Requirements
- **GPU T4**, **Internet ON** (git clone).
- **Add data:** `attacked-pth-files` (from NB1) and the pretrained-weights dataset.
  Set `WEIGHTS_DIR` in Cell 3 to your weights path.

### Output — path layout 
Each run writes, under `/kaggle/working/`:
```
{ds}/{detector}/{adv}/{PREFIX}_net_detector_resnet_{ds}_{adv}.pkl   # fitted detector
{ds}/{detector}/{adv}/{PREFIX}_resnet_{ds}_{adv}.npy                # per-layer test scores
```
where `{detector}` ∈ {knn, randomforest, adaboost, xgboost, lightgbm} and `{PREFIX}` ∈
{KNN, RF, AB, XGB, LGBM} respectively. Example:
`cifar10/adaboost/BIM/AB_net_detector_resnet_cifar10_BIM.pkl` and `AB_resnet_cifar10_BIM.npy`.

### Packaging — do this once, after ALL shards are done
Run every (detector × attack × dataset) shard, collect all their `/kaggle/working/{ds}/...`
outputs, and publish them together as a **single dataset named `enad-pkl`**, preserving the
`{ds}/{detector}/{adv}/...` layout above. Ensemble and Demo assets both point their
`supervised` root at this one `enad-pkl` dataset — keep the `.npy` files alongside the `.pkl`
(NB4/NB5 load the test scores from the `.npy`, so don't drop them when packaging).

In [ ]:
import os
import sys
import json
import pickle
import logging
import subprocess

import numpy as np
import torch

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import PredefinedSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (precision_recall_curve, roc_auc_score,
                             accuracy_score, auc)

UPSTREAM = "deep_Mahalanobis_detector"
UPSTREAM_URL = "https://github.com/pokaxpoka/deep_Mahalanobis_detector.git"
DATA_ROOT = "/kaggle/working/data"
TRAIN_CAP = None     
                     
CV_FOLDS  = 5        

models = None          
data_loader = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def setup(seed: int = 0, clone: bool = True):
    global models, data_loader, DEVICE
    if clone and not os.path.exists(UPSTREAM):
        subprocess.run(["git", "clone", "--quiet", UPSTREAM_URL], check=True)
    sys.path.append("./" + UPSTREAM)
    from deep_Mahalanobis_detector import models as _m, data_loader as _dl
    models, data_loader = _m, _dl
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return models, data_loader


CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

def n_classes(ds_name):  return 100 if ds_name == "cifar100" else 10
def n_layers(net_type):  return 5 if net_type == "resnet" else 4


def _logger(name):
    lg = logging.getLogger(name)
    if not lg.handlers:
        lg.setLevel(logging.INFO)
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s: %(message)s", "%H:%M:%S"))
        lg.addHandler(h)
        lg.propagate = False        
    return lg


def aupr(y_true, y_pred, pos_label=1):
    precision, recall, _ = precision_recall_curve(y_true, y_pred, pos_label=pos_label)
    return auc(recall, precision)

import torchvision.transforms as _T

def get_model_transforms(net_type, ds_name, num_classes,
                         weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth"):
    ckpt = os.path.join(weights_dir, f"{net_type}_{ds_name}.pth")
    if net_type == "resnet":
        model = models.ResNet34(num_c=num_classes)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        tf = _T.Compose([_T.ToTensor(), _T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    else:
        raise ValueError(f"unsupported net_type: {net_type}")
    model.to(DEVICE).eval()
    return model, tf


def extract_activations(X, model, layer_idx, return_pred=False):
    with torch.no_grad():
        acts = model.intermediate_forward(X, layer_idx)
    acts = acts.view(acts.size(0), acts.size(1), -1).mean(2)
    if return_pred:
        with torch.no_grad():
            y_pred = model(X).argmax(1)
        return acts, y_pred
    return acts


def activations_from_loader(model, layer, loader):
    acts, labels = [], []
    for x, y in loader:
        acts.append(extract_activations(x.to(DEVICE), model, layer).cpu().numpy())
        labels.append(y.cpu().numpy())
    return np.concatenate(acts), np.concatenate(labels)

class Datasets:
    def __init__(self, ds_name, in_transform, net_type, adv_type, outf, batch_size=100):
        self.ds_name, self.net_type, self.adv_type = ds_name, net_type, adv_type
        self.train_loader, _ = data_loader.getTargetDataSet(
            ds_name, batch_size, in_transform, DATA_ROOT)
        if TRAIN_CAP and len(self.train_loader.dataset) > TRAIN_CAP:
            from torch.utils.data import DataLoader as _DL, Subset as _Subset
            _idx = np.random.RandomState(0).choice(
                len(self.train_loader.dataset), TRAIN_CAP, replace=False)
            self.train_loader = _DL(_Subset(self.train_loader.dataset, _idx),
                                    batch_size=batch_size, shuffle=False)

        tag = f"{net_type}_{ds_name}_{adv_type}"
        clean = torch.load(f"{outf}/clean_data_{tag}.pth", map_location="cpu")
        new_size = batch_size * (len(clean) // batch_size)
        clean = clean[:new_size]
        noisy = torch.load(f"{outf}/noisy_data_{tag}.pth", map_location="cpu")[:new_size]
        adv = torch.load(f"{outf}/adv_data_{tag}.pth", map_location="cpu")[:new_size]
        targets = torch.load(f"{outf}/label_{tag}.pth", map_location="cpu").numpy()[:new_size]

        self.X_test = torch.cat([adv, clean, noisy]).to(DEVICE)
        self.y_test = np.tile(targets, 3)
        self.adv_test = np.array([0] * len(adv) + [1] * (len(clean) + len(noisy)))

        p_size = len(clean)
        p_split = int(p_size * 0.1)
        idxs_trainval = np.concatenate([
            np.arange(p_split),
            np.arange(p_size, p_size + p_split),
            np.arange(2 * p_size, 2 * p_size + p_split)])
        self.idxs_test = np.delete(np.arange(len(self.X_test)), idxs_trainval)
        pivot = int(len(idxs_trainval) / 6)
        self.idxs_train = np.concatenate([idxs_trainval[:pivot],
                                          idxs_trainval[2*pivot:3*pivot],
                                          idxs_trainval[4*pivot:5*pivot]])
        self.idxs_val = np.concatenate([idxs_trainval[pivot:2*pivot],
                                        idxs_trainval[3*pivot:4*pivot],
                                        idxs_trainval[5*pivot:]])


class DatasetsGA(Datasets):
    def __init__(self, ds_name, in_transform, net_type, adv_type, outf, batch_size=100):
        super().__init__(ds_name, in_transform, net_type, adv_type, outf, batch_size)
        split_size = len(self.X_test) // 3
        (self.idxs_train, self.idxs_val,
         self.idxs_ga_val, self.idxs_test) = self._split(split_size)

    @staticmethod
    def _split(split_size, trainval_frac=0.1, ga_frac=0.1):
        tv = int(split_size * trainval_frac)
        ga = int(split_size * ga_frac)
        idxs_trainval = np.concatenate([
            np.arange(tv),
            np.arange(split_size, split_size + tv),
            np.arange(2*split_size, 2*split_size + tv)])
        idxs_ga_val = np.concatenate([
            np.arange(tv, tv + ga),
            np.arange(split_size + tv, split_size + tv + ga),
            np.arange(2*split_size + tv, 2*split_size + tv + ga)])
        used = np.concatenate([idxs_trainval, idxs_ga_val])
        idxs_test = np.delete(np.arange(split_size * 3), used)
        pivot = int(len(idxs_trainval) / 6)
        idxs_train = np.concatenate([idxs_trainval[:pivot],
                                     idxs_trainval[2*pivot:3*pivot],
                                     idxs_trainval[4*pivot:5*pivot]])
        idxs_val = np.concatenate([idxs_trainval[pivot:2*pivot],
                                   idxs_trainval[3*pivot:4*pivot],
                                   idxs_trainval[5*pivot:]])
        return idxs_train, idxs_val, idxs_ga_val, idxs_test

class _ActLoader:
    def __init__(self, model, ds, idxs_attr, batch_size=100):
        self.model, self.ds, self.idxs_attr, self.bs = model, ds, idxs_attr, batch_size
    def __call__(self, layer_idx):
        X = self.ds.X_test[getattr(self.ds, self.idxs_attr)]
        acts, y = [], []
        for batch in torch.split(X, self.bs):
            a, yp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            acts.append(a.cpu().numpy()); y.append(yp.cpu().numpy())
        return np.concatenate(acts), np.concatenate(y)

def LabelledTrainLoader(model, ds, batch_size=100): return _ActLoader(model, ds, "idxs_train", batch_size)
def LabelledValLoader(model, ds, batch_size=100):   return _ActLoader(model, ds, "idxs_val", batch_size)
def LabelledGAValLoader(model, ds, batch_size=100): return _ActLoader(model, ds, "idxs_ga_val", batch_size)
def LabelledTestLoader(model, ds, batch_size=100):  return _ActLoader(model, ds, "idxs_test", batch_size)

class TrainValLoader:
    def __init__(self, model, ds, batch_size=100):
        self.model, self.ds, self.bs = model, ds, batch_size
    def __call__(self, layer_idx):
        X_train, y_train = activations_from_loader(self.model, layer_idx, self.ds.train_loader)
        adv_train = np.repeat(1, len(X_train))
        X_valid = self.ds.X_test[self.ds.idxs_val]
        av, yv = [], []
        for batch in torch.split(X_valid, self.bs):
            a, yp = extract_activations(batch, self.model, layer_idx, return_pred=True)
            av.append(a.cpu().numpy()); yv.append(yp.cpu().numpy())
        X_valid = np.concatenate(av); y_valid = np.concatenate(yv)
        adv_valid = self.ds.adv_test[self.ds.idxs_val]
        return X_train, X_valid, y_train, y_valid, adv_train, adv_valid

class GroupedScaler(BaseEstimator, TransformerMixin):
    def __init__(self, with_centering=True):
        self.with_centering = with_centering
    def fit(self, X, y=None):
        groups = X[:, -1].astype(int); X = X[:, :-1]
        if self.with_centering:
            self.group_means = [np.mean(X[(groups == lab).nonzero()[0]], axis=0)
                                for lab in np.unique(groups)]
        return self
    def transform(self, X, y=None):
        groups = X[:, -1].astype(int); X = X[:, :-1]
        if not self.with_centering:
            return X
        old_idxs, X_norm = [], []
        for lab in np.unique(groups):
            m = (groups == lab).nonzero()[0]
            X_norm.extend(X[m] - self.group_means[lab]); old_idxs.extend(m)
        return np.array(X_norm)[np.argsort(old_idxs)]
    def get_params(self, deep=True): return {"with_centering": self.with_centering}
    def set_params(self, **p):
        for k, v in p.items(): setattr(self, k, v)
        return self

class _BaseTrainer:
    def __init__(self, n_layers, detector_class, exp_name="", outf="",
                 logger=None, pre_computed=False, pre_computed_path=None):
        self.n_layers = n_layers
        self.detector_class = detector_class
        self.exp_name = exp_name
        self.outf = outf
        self.logger = logger or _logger(exp_name or "enad")
        self.pre_computed = pre_computed
        self.pre_computed_path = pre_computed_path

    def fit(self, dl_train, dl_unseen_train, adv_unseen_train):
        self.detectors = self._train_layer_detectors(dl_train)
        train_scores = self.get_layer_scores(dl_unseen_train)
        self.lr = self._train_logistic(train_scores, adv_unseen_train)
        return self

    def predict(self, data_loader):
        scores = self.get_layer_scores(data_loader)
        preds = self.lr.predict(scores)
        probas = self.lr.predict_proba(scores)
        return scores, np.array([preds, probas[:, 0]]).T

    def _train_logistic(self, X, adv):
        lr = LogisticRegressionCV(penalty="l1", solver="liblinear",
                                  max_iter=10000, n_jobs=-1)
        lr.fit(X, adv)
        return lr

class NetDetector(_BaseTrainer):
    def _train_layer_detectors(self, data_loader):
        detectors = []
        for li in range(self.n_layers):
            X_train, X_valid, y_train, y_valid, adv_train, adv_valid = data_loader(li)
            X = np.concatenate((X_train, X_valid)); y = np.concatenate((y_train, y_valid))
            adv = np.concatenate((adv_train, adv_valid))
            if self.pre_computed:
                with open(self.pre_computed_path) as f:
                    params = json.load(f)
                det = clone(self.detector_class).set_params(
                    **params[f"{self.exp_name}_{li}"]).fit(np.c_[X_train, y_train])
            else:
                from skopt.callbacks import DeltaYStopper
                srch = clone(self.detector_class)
                srch.fit(np.c_[X, y], adv, callback=DeltaYStopper(0.02, n_best=10))
                with open(f"{self.outf}/bayes_{self.exp_name}_{li}.pkl", "wb") as f:
                    pickle.dump(srch, f, pickle.HIGHEST_PROTOCOL)
                det = srch.estimator.set_params(**srch.best_params_).fit(np.c_[X_train, y_train])
                self.logger.info(f"{self.exp_name} L{li}: {srch.best_params_}")
            detectors.append(det)
        return detectors

    def get_layer_scores(self, data_loader):
        scores = []
        for li in range(self.n_layers):
            X, y = data_loader(li)
            scores.append(self.detectors[li].decision_function(np.c_[X, y]))
        return np.vstack(scores).T


class SupervisedDetector(_BaseTrainer):
    def _train_layer_detectors(self, data_loader):
        detectors = []
        for li in range(self.n_layers):
            X_train, X_valid, y_train, y_valid, adv_train, adv_valid = data_loader(li)
            X = np.concatenate((X_train, X_valid)); y = np.concatenate((y_train, y_valid))
            adv = np.concatenate((adv_train, adv_valid))
            if self.pre_computed and self.pre_computed_path:
                with open(self.pre_computed_path) as f:
                    params = json.load(f)
                det = clone(self.detector_class).set_params(
                    **params.get(f"{self.exp_name}_{li}", {})).fit(np.c_[X, y], adv)
            else:
                srch = clone(self.detector_class)
                srch.set_params(cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42))
                srch.fit(np.c_[X, y], adv)
                with open(f"{self.outf}/bayes_{self.exp_name}_{li}.pkl", "wb") as f:
                    pickle.dump(srch, f, pickle.HIGHEST_PROTOCOL)
                det = srch.estimator.set_params(**srch.best_params_).fit(np.c_[X, y], adv)
                self.logger.info(f"{self.exp_name} L{li}: {srch.best_params_}")
            detectors.append(det)
        return detectors

    def get_layer_scores(self, data_loader):
        scores = []
        for li in range(self.n_layers):
            X, y = data_loader(li)
            det = self.detectors[li]
            classes = list(det.classes_)
            adv_idx = classes.index(0) if 0 in classes else 0   # P(adversarial=class 0)
            scores.append(det.predict_proba(np.c_[X, y])[:, adv_idx])
        return np.vstack(scores).T

def _build_registry():
    from sklearn.svm import OneClassSVM
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
    import xgboost as xgb
    import lightgbm as lgb
    from skopt.space import Integer, Real
    return {

        "ocsvm": dict(kind="oneclass", prefix="OCSVM", n_iter=40,   
                      estimator=OneClassSVM(kernel="rbf"),
                      space={"clf__nu": Real(2**-7, 2**-1, prior="log-uniform", base=2),
                             "clf__gamma": Real(2**-15, 2**5, prior="log-uniform", base=2)}),
        "knn": dict(kind="supervised", prefix="KNN", n_iter=20,     
                    estimator=KNeighborsClassifier(),
                    space={"clf__n_neighbors": Integer(1, 15),
                           "clf__weights": ["uniform", "distance"]}),
        "randomforest": dict(kind="supervised", prefix="RF", n_iter=15,  
                    estimator=RandomForestClassifier(n_estimators=100, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200),
                           "clf__max_depth": Integer(3, 20)}),
        "adaboost": dict(kind="supervised", prefix="AB", n_iter=5,       
                    estimator=AdaBoostClassifier(n_estimators=100, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200)}),
        "xgboost": dict(kind="supervised", prefix="XGB", n_iter=40,       
                    estimator=xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200),
                           "clf__max_depth": Integer(3, 10)}),
        "lightgbm": dict(kind="supervised", prefix="LGBM", n_iter=40,     
                    estimator=lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=0),
                    space={"clf__n_estimators": Integer(50, 200),
                           "clf__max_depth": Integer(3, 10)}),
    }

DEFAULT_N_ITER = {
    "knn": 20, "adaboost": 5, "randomforest": 15,
    "xgboost": 40, "lightgbm": 40, "ocsvm": 40,
}


def run_detector(detector, ds_name, adv_type, net_type="resnet",
                 attacked_root="/kaggle/input/datasets/sealeopard/attacked-pth-files",
                 weights_dir="/kaggle/input/datasets/sealeopard/resnet-pth",
                 out_root="/kaggle/working", n_iter=None, batch_size=100,
                 pre_computed=False, pre_computed_path=None, seed=0, verbose=True):
    
    from skopt import BayesSearchCV

    spec = _build_registry()[detector]
    if n_iter is None:
        n_iter = DEFAULT_N_ITER.get(detector, 25)
    outf_attacked = f"{attacked_root}/{ds_name.upper()}/{adv_type}"
    out_dir = f"{out_root}/{ds_name}/{detector}/{adv_type}"
    os.makedirs(out_dir, exist_ok=True)
    exp = f"{net_type}_{ds_name}_{adv_type}"

    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.set_device(0)
        torch.cuda.manual_seed_all(seed)

    model, tf = get_model_transforms(net_type, ds_name, n_classes(ds_name), weights_dir)
    ds = DatasetsGA(ds_name, tf, net_type, adv_type, outf_attacked, batch_size)

    clf = Pipeline([("scaler", GroupedScaler()),
                    ("PCA", PCA(whiten=True, random_state=seed)),
                    ("clf", spec["estimator"])])

    if spec["kind"] == "oneclass":
        if pre_computed:
            layer_det = clf
        else:
            split = PredefinedSplit([-1] * len(ds.train_loader.dataset)
                                    + [1] * len(ds.idxs_val))
            layer_det = BayesSearchCV(clf, spec["space"], n_iter=n_iter, n_points=1,
                                      n_jobs=-1, scoring="accuracy", cv=split,
                                      return_train_score=False, refit=False,
                                      random_state=seed, verbose=0)
        trainer = NetDetector(n_layers(net_type), layer_det, exp_name=exp,
                              outf=out_dir, pre_computed=pre_computed,
                              pre_computed_path=pre_computed_path)
    else:
        if pre_computed:
            layer_det = clf
        else:
            layer_det = BayesSearchCV(clf, spec["space"], n_iter=n_iter, n_points=1,
                                      n_jobs=-1, scoring="accuracy",
                                      return_train_score=False, refit=False,
                                      random_state=seed, verbose=0)
        trainer = SupervisedDetector(n_layers(net_type), layer_det, exp_name=exp,
                                     outf=out_dir, pre_computed=pre_computed,
                                     pre_computed_path=pre_computed_path)

    trainer.fit(TrainValLoader(model, ds, batch_size),
                LabelledTrainLoader(model, ds, batch_size),
                ds.adv_test[ds.idxs_train])

    test_scores, output = trainer.predict(LabelledTestLoader(model, ds, batch_size))
    all_output = np.hstack((test_scores, output,
                            ds.adv_test[ds.idxs_test][:, None]))

    prefix = spec["prefix"]
    with open(f"{out_dir}/{prefix}_net_detector_{exp}.pkl", "wb") as f:
        pickle.dump(trainer, f, pickle.HIGHEST_PROTOCOL)
    np.save(f"{out_dir}/{prefix}_{exp}.npy", all_output)

    acc = accuracy_score(ds.adv_test[ds.idxs_test], output[:, 0])
    auroc = roc_auc_score(ds.adv_test[ds.idxs_test], -output[:, 1])
    if verbose:
        print(f"[{detector}] {exp}: ACC={acc:.4f}  AUROC={auroc*100:.4f}  -> {out_dir}")
    return out_dir, acc, auroc
    
import re

def select_best_lid_maha(ds_name, adv_type, net_type="resnet",
                         numpy_root="/kaggle/input/datasets/sealeopard/mahalanobis-and-lid-numpy",
                         out_root="/kaggle/working", verbose=True):
    src = f"{numpy_root}/{net_type}_{ds_name}/{adv_type}"
    out_dir = f"{out_root}/best_{ds_name}"
    os.makedirs(out_dir, exist_ok=True)
    nl = n_layers(net_type)
    results = {}
    for method in ["Mahalanobis", "LID"]:
        regex = r"(\d+)" if method == "LID" else r"(\d+\.\d+|\d+)"
        params = re.findall(f"{method}_{regex}_{ds_name}_{adv_type}",
                            "".join(os.listdir(src)))
        params = sorted(set(params), key=lambda p: float(p))
        best_auc, best_p = -1.0, None
        for p in params:
            data = np.load(f"{src}/{method}_{p}_{ds_name}_{adv_type}.npy")
            adv = np.select([data[:, -1] == 0, data[:, -1] == 1], [0, 1])
            idxs_tr, idxs_val, _, _ = DatasetsGA._split(len(data) // 3)
            lr = LogisticRegressionCV(penalty="l1", solver="liblinear",
                                      max_iter=10000, n_jobs=-1)
            lr.fit(data[idxs_tr][:, :nl], adv[idxs_tr])
            conf = lr.predict_proba(data[idxs_val][:, :nl])[:, 0]
            a = roc_auc_score(adv[idxs_val], -conf)
            if a > best_auc:
                best_auc, best_p = a, p
        X = np.load(f"{src}/{method}_{best_p}_{ds_name}_{adv_type}.npy")
        np.save(f"{out_dir}/{method}_best_{ds_name}_{adv_type}.npy", X)
        results[method] = (best_p, best_auc)
        if verbose:
            print(f"  {method:12s} best param={best_p}  val AUROC={best_auc*100:.4f}%")
    return out_dir, results

In [2]:
import warnings; warnings.filterwarnings("ignore")   
setup()
print("Device:", DEVICE)

Device: cuda


In [ ]:
DETECTOR = "knn"      # knn | randomforest | adaboost | xgboost | lightgbm
ADV_TYPE = "FGSM"     # FGSM | BIM | DeepFool | CWL2
N_ITER   = 20         # knn 20 | rf 15 | xgb/lgbm 40 | adaboost 5  
SEED = 42

DS_NAME     = "cifar10"
NET_TYPE    = "resnet"
ATTACKED    = "/kaggle/input/datasets/sealeopard/attacked-pth-files"
WEIGHTS_DIR = "/kaggle/input/datasets/sealeopard/resnet-pth"   # <-- set to YOUR weights path

out_dir, acc, auroc = run_detector(
    detector=DETECTOR, ds_name=DS_NAME, adv_type=ADV_TYPE, net_type=NET_TYPE,
    attacked_root=ATTACKED, weights_dir=WEIGHTS_DIR,
    out_root="/kaggle/working", n_iter=N_ITER, batch_size=100, pre_computed=False, seed=SEED,
)
print(f"\nDONE  {DETECTOR}/{ADV_TYPE}:  ACC={acc:.4f}  AUROC={auroc*100:.4f}%  -> {out_dir}")

100%|██████████| 170M/170M [25:35<00:00, 111kB/s]
16:05:15: resnet_cifar10_FGSM L0: OrderedDict({'clf__n_neighbors': 7, 'clf__weights': 'distance'})
16:08:21: resnet_cifar10_FGSM L1: OrderedDict({'clf__n_neighbors': 13, 'clf__weights': 'distance'})
16:12:43: resnet_cifar10_FGSM L2: OrderedDict({'clf__n_neighbors': 4, 'clf__weights': 'uniform'})
16:19:39: resnet_cifar10_FGSM L3: OrderedDict({'clf__n_neighbors': 2, 'clf__weights': 'uniform'})
16:31:06: resnet_cifar10_FGSM L4: OrderedDict({'clf__n_neighbors': 1, 'clf__weights': 'distance'})


[knn] resnet_cifar10_FGSM: ACC=0.9978  AUROC=99.96  -> /kaggle/working/cifar10/knn/FGSM

DONE  knn/FGSM:  ACC=0.9978  AUROC=99.96%  -> /kaggle/working/cifar10/knn/FGSM


### Instantiating the 20 shards
Copy & Edit this notebook; in the **Run** cell set `DETECTOR` and `ADV_TYPE`:

| DETECTOR | ADV_TYPE values to run |
|---|---|
| knn | FGSM, BIM, DeepFool, CWL2 |
| randomforest | FGSM, BIM, DeepFool, CWL2 |
| adaboost | FGSM, BIM, DeepFool, CWL2 |
| xgboost | FGSM, BIM, DeepFool, CWL2 |
| lightgbm | FGSM, BIM, DeepFool, CWL2 |

**Per run:** Save Version (commit, so it runs in the background — don't rely on the interactive
session). When a detector's four attacks are done, gather their outputs into one dataset
`enad-{detector}-pkl` (keep the `{ds}/{detector}/{adv}/...` folders).